In [ ]:
# 1. 原始数据：修改 openml_id，完整显示所有行、特征列和标签列。
from pathlib import Path

import numpy as np
import openml
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

openml_id = 40496
dataset = openml.datasets.get_dataset(openml_id, download_data=False)
X, y, _, _ = dataset.get_data(
    dataset_format="dataframe", target=dataset.default_target_attribute
)
display(pd.concat([X, y], axis=1))
print(f"Dataset ID: {openml_id}")
print(f"样本数: {len(X)}，cluster 数（标签类别数）: {y.nunique()}，feature 数: {X.shape[1]}")

In [ ]:
# 2. 预处理数据：修改 openml_id，读取已保存的 34 个数据集之一。
# processed_features 是缺失值填补、数值缩放和类别独热编码后的特征，尚未进入 ZEUS。
openml_id = 61

with np.load(root / "data" / "embeddings" / f"openml_{openml_id}.npz") as saved:
    processed = pd.DataFrame(saved["processed_features"])
    processed.columns = [f"feature_{i}" for i in range(processed.shape[1])]
    processed["label"] = saved["labels"]

display(processed)


In [ ]:
# 3. 汇总 34 个数据集：样本统计来自实验缓存，ARI 来自现有结果文件。
# LAPD 使用 known K；预测不完整的数据集，其 ARI 保留为 NaN。
kmeans = pd.read_csv(root / "results" / "zeus_openml_baseline.csv")
kmeans = kmeans.loc[kmeans["method"] == "scaled_zeus", ["openml_id", "ari"]]
kmeans = kmeans.rename(columns={"ari": "kmeans_ari"})
cpp = pd.read_csv(root / "results" / "cpp_zeus_openml.csv")
cpp = cpp[["openml_id", "ari"]].rename(columns={"ari": "cpp_ari"})
lapd = pd.read_csv(root / "results" / "lapd_stage4_audit.csv")
lapd = lapd.loc[lapd["mode"] == "known", ["openml_id", "ari"]]
lapd = lapd.rename(columns={"ari": "lapd_ari"})

rows = []
for dataset_id in sorted(kmeans["openml_id"]):
    with np.load(root / "data" / "embeddings" / f"openml_{dataset_id}.npz") as saved:
        class_counts = pd.Series(saved["labels"]).value_counts()
        n_features = saved["processed_features"].shape[1]
    rows.append({
        "openml_id": dataset_id,
        "n_samples": int(class_counts.sum()),
        "n_features": n_features,
        "n_classes": len(class_counts),
        "mean_samples_per_class": class_counts.mean(),
        "min_samples_per_class": int(class_counts.min()),
        "max_samples_per_class": int(class_counts.max()),
    })

summary = pd.DataFrame(rows)
summary = summary.merge(kmeans, on="openml_id", how="left")
summary = summary.merge(cpp, on="openml_id", how="left")
summary = summary.merge(lapd, on="openml_id", how="left")
summary = summary.loc[summary["kmeans_ari"] < 0.5]
display(summary)
print("K-means ARI < 0.5 的数据集 ID:")
print(summary.loc[summary["kmeans_ari"] < 0.5, "openml_id"].tolist())
print(len(summary.loc[summary["n_classes"] == 2]))


In [ ]:
# 4. 修改 openml_id，估计 ZEUS 缩放表征的 intrinsic dimension。
# TWO-NN：使用第一、第二近邻的距离比，采用偏差修正的最大似然估计。
# 方法来源：https://www.nature.com/articles/s41598-022-20991-1
from sklearn.neighbors import NearestNeighbors

openml_id = 53

with np.load(root / "data" / "embeddings" / f"openml_{openml_id}.npz") as saved:
    features = saved["embeddings_scaled"].astype(np.float64)

# 去除完全重复的向量，避免最近邻距离为零；不改变表征的缩放方式。
unique_features = np.unique(features, axis=0)
# 不传入查询数据时，kneighbors 会排除样本自身。
distances, _ = NearestNeighbors(n_neighbors=2).fit(unique_features).kneighbors()
ratios = distances[:, 1] / distances[:, 0]
intrinsic_dim = (len(ratios) - 1) / np.log(ratios).sum()

print(f"OpenML ID: {openml_id}，输入: embeddings_scaled")
print(f"样本数: {len(features)}，去重后: {len(unique_features)}，表征维度: {features.shape[1]}")
print(f"Intrinsic dimension（TWO-NN 估计）: {intrinsic_dim:.3f}")


In [4]:
# 5. 修改 dataset_ids，比较选定数据集的特征数和两种表征的内在维度。
# n_features 为预处理后的特征数，与前面的汇总表一致。
from sklearn.neighbors import NearestNeighbors

dataset_ids = [51, 53, 481, 694, 721, 796, 820, 840, 1495, 1523, 40496, 40705]


def estimate_intrinsic_dimension(features):
    features = np.unique(features.astype(np.float64), axis=0)
    distances, _ = NearestNeighbors(n_neighbors=2).fit(features).kneighbors()
    ratios = distances[:, 1] / distances[:, 0]
    return (len(ratios) - 1) / np.log(ratios).sum()


rows = []
for dataset_id in dataset_ids:
    with np.load(root / "data" / "embeddings" / f"openml_{dataset_id}.npz") as saved:
        rows.append({
            "openml_id": dataset_id,
            "n_features": saved["processed_features"].shape[1],
            "embeddings_scaled_id": estimate_intrinsic_dimension(saved["embeddings_scaled"]),
            "processed_features_id": estimate_intrinsic_dimension(saved["processed_features"]),
        })

display(pd.DataFrame(rows).round(3))


,openml_id,n_features,embeddings_scaled_id,processed_features_id
0,51,25,4.161,3.570
1,53,13,3.467,3.882
2,481,14,3.553,3.780
3,694,8,2.377,2.450
4,721,10,5.882,6.116
5,796,36,1.507,1.296
6,820,12,2.951,7.544
7,840,68,1.384,1.119
8,1495,18,4.400,32.701
9,1523,6,4.554,4.494


In [5]:
# 6. 修改 dataset_ids，分析 embeddings_scaled 的近邻距离和同类比例。
from sklearn.neighbors import NearestNeighbors

dataset_ids = [51, 53, 481, 694, 721, 796, 820, 840, 1495, 1523, 40496, 40705]

rows = []
for dataset_id in dataset_ids:
    with np.load(root / "data" / "embeddings" / f"openml_{dataset_id}.npz") as saved:
        features = saved["embeddings_scaled"].astype(np.float64)
        labels = saved["labels"]

    # 不传入查询数据，排除样本自身；保留重复样本。
    distances, neighbors = NearestNeighbors(n_neighbors=10).fit(features).kneighbors()
    first_distances = distances[:, 0]
    tenth_p10, tenth_median, tenth_p90 = np.quantile(distances[:, 9], [0.1, 0.5, 0.9])
    same_class_ratio = (labels[neighbors] == labels[:, None]).mean()

    rows.append({
        "openml_id": dataset_id,
        "nn1_mean": first_distances.mean(),
        "nn10_p10": tenth_p10,
        "nn10_median": tenth_median,
        "nn10_p90": tenth_p90,
        "same_class_ratio_10": same_class_ratio,
        "nn1_median": np.median(first_distances),
    })

neighbor_summary = pd.DataFrame(rows).set_index("openml_id")
display(neighbor_summary.drop(columns="nn1_median").round(3))
print("第一近邻距离的中位数（按数据集）：")
print(neighbor_summary["nn1_median"].round(3).to_string())


,nn1_mean,nn10_p10,nn10_median,nn10_p90,same_class_ratio_10
openml_id,,,,,
51,0.719,0.599,0.886,3.825,0.771
53,0.492,0.362,0.748,4.404,0.736
481,0.809,0.831,1.776,3.737,0.797
694,0.165,0.174,0.270,2.151,0.756
721,1.423,1.887,2.150,2.726,0.781
796,0.234,0.207,1.511,5.608,0.754
820,0.663,0.710,1.641,3.244,0.784
840,0.444,0.677,1.787,4.485,0.813
1495,0.043,0.250,0.448,0.736,0.994


第一近邻距离的中位数（按数据集）：
openml_id
51       0.539
53       0.324
481      0.681
694      0.059
721      1.408
796      0.097
820      0.545
840      0.301
1495     0.000
1523     1.358
40496    0.000
40705    0.134


In [12]:
# 7. 修改 dataset_ids 和 n_neighbors_list，比较不同邻居数下的谱聚类。
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score

dataset_ids = [51, 53, 481, 694, 721, 796, 820, 840, 1495, 1523, 40496, 40705]
n_neighbors_list = [5, 10, 20, 30, 50]

rows = []
for dataset_id in dataset_ids:
    with np.load(root / "data" / "embeddings" / f"openml_{dataset_id}.npz") as saved:
        features = saved["embeddings_scaled"].astype(np.float64)
        labels = saved["labels"]

    row = {"openml_id": dataset_id}
    # 真实标签仅用于确定簇数和计算 ARI。
    for n_neighbors in n_neighbors_list:
        clustering = SpectralClustering(
            n_clusters=len(np.unique(labels)),
            affinity="nearest_neighbors",
            n_neighbors=n_neighbors,
            random_state=42,
            n_init=100,
        )
        predictions = clustering.fit_predict(features)
        row[f"spectral_ari_{n_neighbors}"] = adjusted_rand_score(labels, predictions)
        row[f"n_components_{n_neighbors}"] = connected_components(
            clustering.affinity_matrix_, directed=False, return_labels=False
        )
    rows.append(row)

columns = ["openml_id"] + [
    f"{metric}_{k}"
    for metric in ["spectral_ari", "n_components"]
    for k in n_neighbors_list
]
display(pd.DataFrame(rows)[columns].round(3))


e:\dev\manifold-clustering\.venv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
e:\dev\manifold-clustering\.venv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
e:\dev\manifold-clustering\.venv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
e:\dev\manifold-clustering\.venv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
e:\dev\manifold-clustering\.venv\Lib\site-packages\sklearn\manifold\_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.w

,openml_id,spectral_ari_5,spectral_ari_10,spectral_ari_20,spectral_ari_30,spectral_ari_50,n_components_5,n_components_10,n_components_20,n_components_30,n_components_50
0,51,0.067,-0.010,0.071,0.071,0.071,1,1,1,1,1
1,53,0.249,0.192,0.047,0.349,0.349,5,2,1,1,1
2,481,-0.031,-0.036,-0.012,-0.013,0.076,4,2,1,1,1
3,694,0.277,0.352,0.362,0.340,0.376,7,3,1,1,1
4,721,0.433,0.433,0.433,0.433,0.433,2,2,2,2,2
5,796,0.068,0.064,-0.018,-0.018,0.138,16,6,2,2,1
6,820,0.431,0.298,0.270,0.298,0.298,1,1,1,1,1
7,840,0.062,-0.022,0.001,0.311,0.357,14,3,1,1,1
8,1495,-0.006,-0.007,-0.003,-0.003,0.073,17,4,3,3,3
9,1523,0.200,0.200,0.204,0.211,0.216,1,1,1,1,1
